### Carregamento das bibliotecas

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from utils.parse_date import parse_date
from utils.dollar_sales_quote import get_cambio_util

#### Carregando os dados

In [27]:
df = pd.read_csv("../data/raw/vendas_2023_2024.csv", index_col="id")
df = df.reset_index(drop=True)

df.head(10)

,id_client,id_product,qtd,total,sale_date
0,42,105,11,3405.00,2023-09-10
1,3,136,9,16873.90,15-09-2024
2,25,139,7,9475.30,2024-08-13
3,20,23,5,55893.00,2023-02-03
4,8,57,4,451403.90,2024-02-12
5,36,52,3,39056.40,2023-09-26
6,27,25,3,34560.05,2024-02-28
7,37,26,7,114932.90,07-11-2023
8,31,143,3,12643.55,2024-08-25
9,39,128,5,23254.00,2023-05-07


como já foi feita uma análise previamente, já sabemos que a data tem diferentes tipo de formato "d-m-y" e "y-m-d", vamos padronizar isso para o tipo data

In [28]:
df["sale_date"] = df["sale_date"].apply(parse_date)

In [29]:
df.head(10)

,id_client,id_product,qtd,total,sale_date
0,42,105,11,3405.00,2023-09-10
1,3,136,9,16873.90,2024-09-15
2,25,139,7,9475.30,2024-08-13
3,20,23,5,55893.00,2023-02-03
4,8,57,4,451403.90,2024-02-12
5,36,52,3,39056.40,2023-09-26
6,27,25,3,34560.05,2024-02-28
7,37,26,7,114932.90,2023-11-07
8,31,143,3,12643.55,2024-08-25
9,39,128,5,23254.00,2023-05-07


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id_client   9895 non-null   int64         
 1   id_product  9895 non-null   int64         
 2   qtd         9895 non-null   int64         
 3   total       9895 non-null   float64       
 4   sale_date   9895 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 386.7 KB


In [31]:
datas_unicas = df['sale_date'].unique()
print(f"Total de chamadas à API: {len(datas_unicas)}") 

Total de chamadas à API: 725


In [32]:
cache_cambio = {}

try:
  
  print("Buscando câmbios... (isso pode demorar alguns minutos)")

  for data in datas_unicas:
    cache_cambio[data] = get_cambio_util(pd.Timestamp(data))

  print("Busca concluída!")

except ValueError as e:
  print(f"Erro ao buscar cambios, erro: {e}")

Buscando câmbios... (isso pode demorar alguns minutos)
Busca concluída!


In [33]:
df['cambio'] = df['sale_date'].map(cache_cambio)

In [34]:
df.head(20)

,id_client,id_product,qtd,total,sale_date,cambio
0,42,105,11,3405.00,2023-09-10,4.9835
1,3,136,9,16873.90,2024-09-15,5.5717
2,25,139,7,9475.30,2024-08-13,5.4875
3,20,23,5,55893.00,2023-02-03,5.1030
4,8,57,4,451403.90,2024-02-12,4.9717
5,36,52,3,39056.40,2023-09-26,4.9717
6,27,25,3,34560.05,2024-02-28,4.9557
7,37,26,7,114932.90,2023-11-07,4.8670
8,31,143,3,12643.55,2024-08-25,5.5263
9,39,128,5,23254.00,2023-05-07,4.9696


tudo certo, exportarei para os dados processados para poder trabalhar com ele

In [25]:
df.to_csv('../data/processed/vendas_2023_2024_processed.csv', index=False)